<a href="https://colab.research.google.com/github/cc2872/Human-behavior-ML/blob/gate-1/colab_selfcontained_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BerryWorld difficulty sweep, self-contained

All code is written to disk by the cells below, so there is **nothing to upload** and no stale-file problems.

**To run:** `Runtime -> Change runtime type -> GPU`, then `Runtime -> Run all`. The first cell only *checks* the environment (it never uninstalls or downgrades anything -- that was the old failure mode); the write cells are instant; the sweep cell trains on GPU.

**If the first cell reports a broken runtime:** `Runtime -> Disconnect and delete runtime`, then `Run all`. A poisoned numpy can't be fixed in place by pip; only a fresh kernel clears it.

**The question:** the faithful population run was a clean null (every condition solves poison avoidance equally, enforcement extinct). Diagnosis: at D=25 the individual signal is enough, so the taboo is redundant. This sweep raises the poison delay D; the Koster effect can only appear where direct learning **fails**. Signature: `()` plateaus high while `(0,)` drops below it as D grows.

## Install

In [ ]:
# Environment check -- NON-DESTRUCTIVE. Colab already ships a working
# jax + flax + optax + numpy + scipy. The ONLY thing that ever broke this
# notebook was pinning numpy/scipy DOWN, which corrupts the in-process numpy
# (numpy._core.umath loses _center) and cascades into every later cell.
# So: never uninstall, never pin down. Install only genuinely-missing deps.
import importlib, subprocess, sys

def _have(mod):
    try:
        importlib.import_module(mod); return True
    except Exception:
        return False

missing = [m for m in ("flax", "optax") if not _have(m)]
if missing:
    print("installing (no version pins, numpy/scipy untouched):", missing)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing],
                   check=False)

# Integrity probe. Reproduces the exact import path that used to fail
# (scipy.linalg + a jax linalg op, which flax's GRUCell orthogonal init walks),
# so a runtime already poisoned by an earlier bad install is caught HERE with a
# clear instruction, instead of 40 cells later inside a jitted train step.
try:
    import numpy as np;      np.array([1.0, 2.0]).sum()
    import scipy;            import scipy.linalg          # scipy<->numpy ABI
    import jax, jax.numpy as jnp
    _q, _r = jnp.linalg.qr(jnp.eye(3))                    # GRU init uses this
    import flax, optax
    print("numpy   ", np.__version__)
    print("scipy   ", scipy.__version__)
    print("jax     ", jax.__version__)
    print("flax    ", flax.__version__, "  optax", optax.__version__)
    devs = jax.devices()
    print("devices:", devs)
    if not any(d.platform == "gpu" for d in devs):
        print("\n[!] No GPU visible. Runtime -> Change runtime type -> GPU,"
              " then Runtime -> Run all.")
    else:
        print("\nOK -- GPU ready. Runtime -> Run all.")
except ImportError as e:
    print("[X] This runtime is already in a broken state -- a previous install"
          " corrupted numpy/scipy in place.")
    print("    pip CANNOT repair an already-imported broken numpy in the running"
          " kernel.")
    print("    Fix: Runtime -> Disconnect and delete runtime, then Runtime ->"
          " Run all.")
    print("    (underlying error:", e, ")")


## Write the code files (instant, replaces any old copies)

In [ ]:
%%writefile berryworld.py
"""
berryworld.py -- minimal reference implementation of a Koster-style norm substrate.

NumPy, single file, no dependencies beyond numpy. This is the *debuggable*
version: written for clarity so the game logic can be verified before porting
to JAX. Every array has a fixed shape so the port is mechanical.

MECHANICS
---------
Two berry types on a grid.
  Berry 0 ("poison"): +r_eat now, then -r_poison delayed by D steps.
                      Grounded in the ENVIRONMENT. Avoidance should survive
                      isolation -- this is the positive control.
  Berry 1 ("harmless"): +r_eat now, no consequence ever.
                        Grounded only in the POPULATION, if marked.

Eating berry type t sets a visible mark of type t on the eater for M steps.
Which berry types produce a visible mark is a CONDITION parameter
(`marked_berries`), giving Koster's three conditions:
    ()      -> no rule
    (0,)    -> important rule only
    (0, 1)  -> important + silly rule

Zapping costs the zapper c_zap and the target c_zapped. The environment
does NOT know which marks "deserve" punishment. Who gets zapped is learned.
That is the whole point -- do not build the norm in.

Observations are egocentric windows, flattened. Rewards are per-agent.

SMOKE TEST
----------
  python berryworld.py
"""
import numpy as np

# ------------------------------------------------------------------ config
class Config:
    grid = 15            # square grid, walls on the border
    n_agents = 6
    view = 3             # egocentric half-window -> (2*view+1)^2 cells
    n_berry_types = 2

    cells_per_type = 54  # berry cells per type, equal by construction so
                         # scarcity can't bias consumption. 54 = 6 disjoint
                         # 3x3 blocks, ~ the original type-0 poison pressure.
    # Outcome-neutral BATCH-check thresholds (see outcome_neutral_suite).
    # Per-seed clustering gaps of the fixed and old constructions OVERLAP
    # (fixed |gap| up to 0.41, old down to 0.04), so no per-seed threshold has
    # power. The powered test is on the ACROSS-SEED MEAN gap: null (fixed) is
    # 0.036 +/- 0.030 SEM over 30 seeds; the broken construction sits at 0.50.
    clustering_gap_tol = 0.15  # ~4x null SEM: passes fixed, rejects old (~8 SEM)
    eats_t_tol = 3.0           # |paired t| on eats-by-type under random policy
    val_seeds = 30             # seeds for the batch outcome-neutral suite
    regrow_prob = 0.01   # per-step per-empty-site regrowth within a patch

    r_eat = 1.0
    r_poison = 4.0       # magnitude of the delayed penalty
    poison_delay = 25    # D: steps between eating berry 0 and the penalty
    mark_steps = 40      # M: how long a mark stays visible

    c_zap = 0.1          # cost to the zapper
    c_zapped = 2.0       # cost to the target
    zap_range = 4        # beam length, fires along facing direction
    zap_removal_steps = 25   # framesTillRespawn: a zapped agent is REMOVED from
                             # play this many steps, then respawns at a free cell.
                             # This is the Koster mechanic (Melting Pot Zapper
                             # removeHitPlayer); the lost foraging time is what
                             # gives enforcement a selfish return. 0 = no removal.
    r_zap_bonus = 0.0    # rewardForZapping: direct reward to the zapper on a
                         # landed zap. Kept 0 -- incentive comes from removal.

    marked_berries = (0, 1)   # condition parameter; see docstring
    episode_len = 1000


# actions: 0-3 move N/E/S/W (also sets facing), 4 eat, 5 zap, 6 noop
N_ACTIONS = 7
_DELTA = np.array([[-1, 0], [0, 1], [1, 0], [0, -1]])


class BerryWorld:
    def __init__(self, cfg=Config(), seed=0):
        self.c = cfg
        self.rng = np.random.default_rng(seed)
        self._build_patches()
        self.reset()

    # ------------------------------------------------------------- setup
    @staticmethod
    def _clustering(mask):
        """Mean number of 4-neighbours of each berry cell that share its type.
        An isolated 3x3 block scores 24/9 = 2.67; fragmented remnants score
        lower, touching/overlapping blocks higher. Used to check that the two
        types are structurally comparable, not just equal in count."""
        nb = np.zeros(mask.shape, int)
        nb[1:, :] += mask[:-1, :]
        nb[:-1, :] += mask[1:, :]
        nb[:, 1:] += mask[:, :-1]
        nb[:, :-1] += mask[:, 1:]
        n = int(mask.sum())
        return float(nb[mask].sum()) / n if n else 0.0

    def _build_patches(self):
        """Symmetric patch construction, resampled only on __init__ so
        episodes are comparable across a run. Both berry types are drawn from
        ONE generative process -- alternate type, place a random 3x3 block
        whose footprint is disjoint from every already-claimed cell -- so
        spatial structure (both count AND clustering) is identical by symmetry
        rather than by post-hoc correction. The old "earlier type wins the
        overlap" rule made later types both scarcer and more fragmented, with
        seed-dependent variance; this removes both confounds at the source."""
        c = self.c
        G = c.grid
        T = c.n_berry_types
        target = c.cells_per_type
        self.patch_mask = np.zeros((T, G, G), bool)
        claimed = np.zeros((G, G), bool)        # any-type occupancy
        centres = [[] for _ in range(T)]        # 3x3 block centres per type

        # Phase 1: disjoint 3x3 blocks, alternating type. Centres kept one cell
        # in from the interior edge so each block lands fully inside -> exactly
        # 9 cells, no border clipping. 12 disjoint blocks (target 54) is above
        # the random-packing (RSA) jamming limit for this grid, so one type may
        # place fewer than the other -- Phase 2 equalizes that away.
        want = target // 9                      # full blocks desired per type
        order, stalled = 0, 0
        while any(len(centres[t]) < want for t in range(T)) and stalled < T:
            t = order % T
            order += 1
            if len(centres[t]) >= want:
                continue
            for _ in range(500):                # rejection sampling
                r = int(self.rng.integers(2, G - 2))
                q = int(self.rng.integers(2, G - 2))
                blk = (slice(r - 1, r + 2), slice(q - 1, q + 2))
                if not claimed[blk].any():
                    claimed[blk] = True
                    centres[t].append((r, q))
                    stalled = 0
                    break
            else:                               # no disjoint spot found
                stalled += 1

        # Phase 2: force IDENTICAL composition across types so structure can't
        # depend on placement order. Keep the same number of full blocks for
        # every type (drop extras from whichever type packed more, freeing
        # them), then top every type up to `target` with the same number of
        # loose cells drawn from the shared unclaimed interior pool.
        n_blk = min(len(cs) for cs in centres)
        for t in range(T):
            for (r, q) in centres[t][n_blk:]:   # release surplus blocks
                claimed[r - 1:r + 2, q - 1:q + 2] = False
            for (r, q) in centres[t][:n_blk]:   # paint the kept blocks
                self.patch_mask[t, r - 1:r + 2, q - 1:q + 2] = True
        for t in range(T):
            need = target - int(self.patch_mask[t].sum())
            for _ in range(need):
                free = np.argwhere(~claimed)
                free = free[(free[:, 0] > 0) & (free[:, 0] < G - 1)
                            & (free[:, 1] > 0) & (free[:, 1] < G - 1)]
                if not len(free):
                    break
                r, q = free[int(self.rng.integers(len(free)))]
                self.patch_mask[t, r, q] = True
                claimed[r, q] = True

        # --- exact per-instance structural invariant: equal cells per type.
        # Clustering is a DISTRIBUTIONAL property -- its per-seed gap has no
        # power (fixed and broken constructions overlap seed-to-seed), so it is
        # verified on the across-seed mean in outcome_neutral_suite(), not here.
        counts = self.patch_mask.reshape(c.n_berry_types, -1).sum(1)
        assert (counts == counts[0]).all(), \
            f"invariant violated: unequal cells per type {counts}"
        self.patch_clustering = np.array(
            [self._clustering(self.patch_mask[t])
             for t in range(c.n_berry_types)])  # exposed for the batch check

    def reset(self):
        c = self.c
        self.t = 0
        self.berries = self.patch_mask.copy()          # (T, G, G) bool

        free = np.argwhere(~self.berries.any(0))
        free = free[(free[:, 0] > 0) & (free[:, 0] < c.grid - 1)
                    & (free[:, 1] > 0) & (free[:, 1] < c.grid - 1)]
        idx = self.rng.choice(len(free), c.n_agents, replace=False)
        self.pos = free[idx].copy()                    # (N, 2) int
        self.facing = self.rng.integers(0, 4, c.n_agents)

        self.marks = np.zeros((c.n_agents, c.n_berry_types), int)   # countdown
        # pending[i, k] = steps until the k-th queued poison hit lands, 0 = none
        self.pending = np.zeros((c.n_agents, c.poison_delay + 1), bool)
        # respawn[i] > 0 => agent i is removed from play (zapped); counts down
        self.respawn = np.zeros(c.n_agents, int)
        return self.observe()

    # ------------------------------------------------------- observation
    def observe(self):
        """Egocentric (2v+1)^2 windows, channels:
        [wall, berry0, berry1, agent, mark0, mark1] + self scalars."""
        c, v = self.c, self.c.view
        w = 2 * v + 1
        G = c.grid

        wall = np.zeros((G, G), np.float32)
        wall[0, :] = wall[-1, :] = wall[:, 0] = wall[:, -1] = 1.0

        active = self.respawn == 0                     # removed agents are off-grid
        occ = np.zeros((G, G), np.float32)
        mk = np.zeros((c.n_berry_types, G, G), np.float32)
        for i, (r, q) in enumerate(self.pos):
            if not active[i]:
                continue
            occ[r, q] = 1.0
            for t in range(c.n_berry_types):
                if self.marks[i, t] > 0 and t in c.marked_berries:
                    mk[t, r, q] = 1.0

        planes = np.concatenate(
            [wall[None], self.berries.astype(np.float32), occ[None], mk], 0)
        P = planes.shape[0]
        pad = np.pad(planes, ((0, 0), (v, v), (v, v)), constant_values=0.0)

        obs = np.zeros((c.n_agents, P * w * w + 2 + c.n_berry_types), np.float32)
        for i, (r, q) in enumerate(self.pos):
            if not active[i]:
                continue                               # removed -> all-zero obs
            win = pad[:, r:r + w, q:q + w]             # (P, w, w)
            self_feats = np.concatenate([
                [self.facing[i] / 3.0],
                [self.pending[i].any().astype(np.float32)],   # NOT observable
                (self.marks[i] > 0).astype(np.float32),
            ])
            obs[i] = np.concatenate([win.ravel(), self_feats])
        # zero the pending channel: the agent must infer poisoning from
        # consequences, not read it off the observation. Flip this to 1.0
        # only as a diagnostic.
        obs[:, -1 - self.c.n_berry_types] = 0.0
        return obs

    # ------------------------------------------------------------- step
    def step(self, actions):
        c = self.c
        actions = np.asarray(actions, int)
        rew = np.zeros(c.n_agents, np.float32)
        info = {"eats": np.zeros(c.n_berry_types, int),  # eats by berry type
                "zaps_fired": 0,      # zap actions issued (each costs c_zap)
                "zaps_landed": 0,     # beams that actually hit an agent
                "poison_hits": 0,     # delayed poison penalties that landed
                "zaps_on_marked": 0,  # landed zaps whose target was visibly marked
                "marked_agents": 0,   # agents visibly marked this step (numerator)
                "active_agents": 0}   # agents on-grid this step (prevalence base)

        # --- 1. delayed poison lands first (independent of this step's action)
        info["poison_hits"] = int(self.pending[:, 0].sum())
        rew -= c.r_poison * self.pending[:, 0]
        self.pending = np.roll(self.pending, -1, axis=1)
        self.pending[:, -1] = False

        # active agents only: removed (zapped) agents sit out the whole step
        active = self.respawn == 0
        info["active_agents"] = int(active.sum())

        # --- 2. movement, resolved simultaneously; collisions cancel. Removed
        # agents are off-grid -> excluded from collision via unique sentinel keys.
        mv = (actions < 4) & active
        self.facing = np.where(mv, actions, self.facing)
        target = self.pos.copy()
        target[mv] += _DELTA[actions[mv]]
        target = np.clip(target, 1, c.grid - 2)
        keys = np.where(active, target[:, 0] * c.grid + target[:, 1],
                        -1 - np.arange(c.n_agents))
        _, inv, counts = np.unique(keys, return_inverse=True, return_counts=True)
        ok = counts[inv] == 1
        self.pos[ok] = target[ok]

        # --- 3. eating (active agents only)
        eat = (actions == 4) & active
        for i in np.flatnonzero(eat):
            r, q = self.pos[i]
            for t in range(c.n_berry_types):
                if self.berries[t, r, q]:
                    self.berries[t, r, q] = False
                    rew[i] += c.r_eat
                    self.marks[i, t] = c.mark_steps
                    info["eats"][t] += 1
                    if t == 0:
                        self.pending[i, -1] = True     # lands in D steps
                    break

        # --- 4. zapping: beam along facing, hits the nearest ACTIVE agent. A
        # landed zap removes the target from play for zap_removal_steps (Koster
        # timeout), so the lost foraging time gives enforcement a selfish return.
        zap = (actions == 5) & active
        info["zaps_fired"] = int(zap.sum())
        # visibly-marked agents (the only mark a zapper could condition on):
        # a mark of a type in marked_berries that is still active. Empty in
        # condition () -> nothing to enforce on, by construction.
        vis_marked = np.zeros(c.n_agents, bool)
        for t in c.marked_berries:
            vis_marked |= self.marks[:, t] > 0
        info["marked_agents"] = int((vis_marked & active).sum())
        alive = active.copy()                          # mutated as targets fall
        for i in np.flatnonzero(zap):
            rew[i] -= c.c_zap
            d = _DELTA[self.facing[i]]
            for k in range(1, c.zap_range + 1):
                cell = self.pos[i] + d * k
                hit = np.flatnonzero(
                    alive & (self.pos[:, 0] == cell[0])
                    & (self.pos[:, 1] == cell[1]))
                if len(hit):
                    tgt = hit[0]
                    rew[tgt] -= c.c_zapped
                    rew[i] += c.r_zap_bonus
                    info["zaps_landed"] += 1
                    if vis_marked[tgt]:
                        info["zaps_on_marked"] += 1
                    if c.zap_removal_steps > 0:
                        self.respawn[tgt] = c.zap_removal_steps
                        alive[tgt] = False             # off-grid immediately
                    break

        # --- 5. regrowth, respawn, and bookkeeping
        empty = self.patch_mask & ~self.berries
        self.berries |= empty & (
            self.rng.random(self.berries.shape) < c.regrow_prob)

        # respawn: tick removal timers; agents whose timer reaches 0 reappear at
        # a free interior cell (no berry, no active agent).
        respawning = self.respawn == 1
        self.respawn = np.maximum(self.respawn - 1, 0)
        if respawning.any():
            blocked = np.zeros((c.grid, c.grid), bool)
            on_grid = (self.respawn == 0) & ~respawning
            blocked[self.pos[on_grid, 0], self.pos[on_grid, 1]] = True
            for i in np.flatnonzero(respawning):
                free = np.argwhere(~self.berries.any(0) & ~blocked)
                free = free[(free[:, 0] > 0) & (free[:, 0] < c.grid - 1)
                            & (free[:, 1] > 0) & (free[:, 1] < c.grid - 1)]
                if not len(free):
                    self.respawn[i] = 1                # no room; retry next step
                    continue
                r, q = free[int(self.rng.integers(len(free)))]
                self.pos[i] = (r, q)
                self.facing[i] = int(self.rng.integers(0, 4))
                blocked[r, q] = True

        self.marks = np.maximum(self.marks - 1, 0)

        self.t += 1
        done = self.t >= c.episode_len
        return self.observe(), rew, done, info


# ------------------------------------------------- outcome-neutral batch suite
def outcome_neutral_suite(n_seeds=Config.val_seeds, verbose=True):
    """Checks that require a distribution over seeds rather than one
    construction. Pre-specified, orthogonal to any norm/enforcement hypothesis,
    and calibrated to have POWER against the specific failure they guard:

      clustering gap  -- no SYSTEMATIC spatial-structure difference between
                         types. Tested on the across-seed MEAN because the
                         per-seed gap can't separate fixed from broken (their
                         distributions overlap). Null mean 0.036 +/- 0.030 SEM;
                         broken construction sits at ~0.50 -> rejected at ~8 SEM.
      eats by type    -- the DIRECT target the structural checks are proxies
                         for: a null (random) policy must consume both types at
                         indistinguishable rates. Paired t over seeds; unequal
                         counts (the old build) drive |t| far past the tol.

    Returns the computed statistics so they can be logged into the protocol.
    """
    gaps, e0, e1 = [], [], []
    for s in range(n_seeds):
        env = BerryWorld(seed=s)             # equal-count assert fires here
        cl = env.patch_clustering
        gaps.append(cl[0] - cl[1])
        env.reset()
        rng = np.random.default_rng(10_000 + s)
        tot = np.zeros(env.c.n_berry_types, int)
        for _ in range(env.c.episode_len):
            a = rng.integers(0, N_ACTIONS, env.c.n_agents)
            _, _, done, info = env.step(a)
            tot += info["eats"]
            if done:
                break
        e0.append(tot[0]); e1.append(tot[1])

    gaps = np.asarray(gaps, float)
    e0, e1 = np.asarray(e0, float), np.asarray(e1, float)
    diff = e0 - e1
    gap_mean = float(gaps.mean())
    gap_sem = float(gaps.std(ddof=1) / np.sqrt(n_seeds))
    eats_t = float(diff.mean() / (diff.std(ddof=1) / np.sqrt(n_seeds)))

    if verbose:
        print(f"\n=== outcome-neutral batch suite ({n_seeds} seeds) ===")
        print(f"clustering gap (0-1) mean {gap_mean:+.3f}  SEM {gap_sem:.3f}"
              f"   |tol| {Config.clustering_gap_tol}")
        print(f"eats/type  poison {e0.mean():.1f}  harmless {e1.mean():.1f}"
              f"   paired t {eats_t:+.2f}  |tol| {Config.eats_t_tol}")

    assert abs(gap_mean) < Config.clustering_gap_tol, (
        f"systematic clustering gap {gap_mean:+.3f} "
        f">= tol {Config.clustering_gap_tol}")
    assert abs(eats_t) < Config.eats_t_tol, (
        f"eats-by-type asymmetry under random policy: paired t={eats_t:+.2f}")
    if verbose:
        print("batch checks OK      no systematic clustering gap; "
              "eats-by-type indistinguishable")
    return {"gap_mean": gap_mean, "gap_sem": gap_sem, "eats_t": eats_t,
            "eats_poison": float(e0.mean()), "eats_harmless": float(e1.mean())}


# ------------------------------------------------------------ smoke test
if __name__ == "__main__":
    env = BerryWorld(seed=0)
    obs = env.reset()
    print("obs shape          ", obs.shape)
    print("berries at reset   ", env.berries.sum(axis=(1, 2)))
    print("patch clustering   ", np.round(env.patch_clustering, 2))

    rng = np.random.default_rng(1)
    total = np.zeros(env.c.n_agents)
    eats = np.zeros(env.c.n_berry_types, int)
    zaps_fired = zaps_landed = poison_hits = 0
    for step in range(env.c.episode_len):
        a = rng.integers(0, N_ACTIONS, env.c.n_agents)
        obs, r, done, info = env.step(a)
        eats += info["eats"]
        zaps_fired += info["zaps_fired"]
        zaps_landed += info["zaps_landed"]
        poison_hits += info["poison_hits"]
        total += r
        if done:
            break

    pending_at_end = int(env.pending.sum())

    print("random-policy return", np.round(total, 2))
    print("eats by type        ", eats)
    print("zaps fired / landed ", zaps_fired, "/", zaps_landed)
    print("poison hits landed  ", poison_hits)
    print("poison still pending", pending_at_end)
    print("marks still active  ", (env.marks > 0).sum(0))

    # outcome-neutral BEHAVIOURAL invariant #1: poison bookkeeping. Each
    # type-0 eat queues exactly one delayed hit, so
    #   eats[0] == hits that landed + hits still in flight at episode end.
    assert eats[0] == poison_hits + pending_at_end, (
        f"poison mismatch: {eats[0]} != {poison_hits} + {pending_at_end}")
    print(f"poison check OK      {eats[0]} eaten == "
          f"{poison_hits} landed + {pending_at_end} pending")

    # outcome-neutral BEHAVIOURAL invariant #2: the four reward channels
    # reconstruct the total return with zero residual (no unlogged reward).
    recon = (eats.sum() * env.c.r_eat
             - poison_hits * env.c.r_poison
             - zaps_landed * env.c.c_zapped
             - zaps_fired * env.c.c_zap
             + zaps_landed * env.c.r_zap_bonus)
    assert abs(recon - total.sum()) < 1e-4, (
        f"reward residual {recon - total.sum():.4f}: channels don't close")
    print(f"reward check OK      channels reconstruct {total.sum():.1f} "
          f"with 0 residual")

    # structural + behavioural checks that need a distribution over seeds
    outcome_neutral_suite()

    print("\nsanity: return should be near zero or negative under a random")
    print("policy -- eating is rare, zapping is frequent and costly.")



In [ ]:
%%writefile berryworld_jax.py
"""
berryworld_jax.py -- pure-JAX port of berryworld.py, diffed against the NumPy
oracle (berryworld.BerryWorld). Fixed array shapes, no python control flow in
the stepped path, so it jits and vmaps over seeds/agents on device.

Design (see project_brief.md 3.1, 7):
  * Patch construction stays HOST-side (NumPy oracle builds patch_mask); the
    JAX env takes it as a static-shaped input. The hard combinatorial part is
    not re-derived here.
  * Removal/respawn are MASKS (active, respawn timer), never reshapes.
  * The order-dependent zap (NumPy mutates `alive` as targets fall) is
    replicated with lax.scan over agents in index order, so beam resolution
    matches the oracle rather than approximating it.

Correctness is established by oracle diff, not by re-reading this file: see
diff_jax_oracle.py. Deterministic dynamics match exactly; stochastic draws
(regrowth, respawn placement) use JAX PRNG and are checked distributionally.
"""
from functools import partial
from typing import NamedTuple
import jax
import jax.numpy as jnp
from jax import lax

N_ACTIONS = 7
_DELTA = jnp.array([[-1, 0], [0, 1], [1, 0], [0, -1]])   # N, E, S, W


class JCfg(NamedTuple):
    """Static config (python scalars -> hashable -> static_argnums)."""
    grid: int = 15
    n_agents: int = 6
    view: int = 3
    n_berry_types: int = 2
    poison_delay: int = 25
    mark_steps: int = 40
    zap_range: int = 4
    zap_removal_steps: int = 25
    episode_len: int = 300
    r_eat: float = 1.0
    r_poison: float = 4.0
    c_zap: float = 0.1
    c_zapped: float = 2.0
    r_zap_bonus: float = 0.0
    regrow_prob: float = 0.01
    marked_mask: tuple = (True, True)     # per-type: does eating it show a mark


class State(NamedTuple):
    berries: jnp.ndarray      # (T, G, G) bool
    pos: jnp.ndarray          # (N, 2) int32
    facing: jnp.ndarray       # (N,) int32
    marks: jnp.ndarray        # (N, T) int32   countdown
    pending: jnp.ndarray      # (N, D+1) bool
    respawn: jnp.ndarray      # (N,) int32     0 = active
    patch_mask: jnp.ndarray   # (T, G, G) bool  static regrow template
    t: jnp.ndarray            # scalar int32
    key: jnp.ndarray          # PRNG key


def _cell_key(pos, G):
    return pos[:, 0] * G + pos[:, 1]


@partial(jax.jit, static_argnums=(0,))
def observe(cfg: JCfg, s: State):
    """Egocentric (2v+1)^2 windows, channels [wall, berry0, berry1, agent,
    mark0, mark1] + [facing, pending(zeroed), mark0>0, mark1>0]."""
    G, v, T, N = cfg.grid, cfg.view, cfg.n_berry_types, cfg.n_agents
    w = 2 * v + 1
    active = s.respawn == 0

    wall = jnp.zeros((G, G), jnp.float32)
    wall = wall.at[0, :].set(1.).at[-1, :].set(1.).at[:, 0].set(1.).at[:, -1].set(1.)

    occ = jnp.zeros((G, G), jnp.float32)
    occ = occ.at[s.pos[:, 0], s.pos[:, 1]].add(active.astype(jnp.float32))

    marked_mask = jnp.array(cfg.marked_mask)                 # (T,)
    vis = (s.marks > 0) & marked_mask[None, :] & active[:, None]   # (N, T)
    mk = jnp.zeros((T, G, G), jnp.float32)
    for t in range(T):
        mk = mk.at[t, s.pos[:, 0], s.pos[:, 1]].add(vis[:, t].astype(jnp.float32))

    planes = jnp.concatenate(
        [wall[None], s.berries.astype(jnp.float32), occ[None], mk], 0)   # (P,G,G)
    P = planes.shape[0]
    pad = jnp.pad(planes, ((0, 0), (v, v), (v, v)))

    def window(p):                                           # p = (r, q)
        return lax.dynamic_slice(pad, (0, p[0], p[1]), (P, w, w))
    wins = jax.vmap(window)(s.pos).reshape(N, -1)            # (N, P*w*w)

    self_feats = jnp.concatenate([
        (s.facing / 3.0)[:, None],
        jnp.zeros((N, 1), jnp.float32),                      # pending: NOT observable
        (s.marks > 0).astype(jnp.float32),
    ], axis=1)                                               # (N, 2+T)
    obs = jnp.concatenate([wins, self_feats], axis=1)
    return obs * active[:, None].astype(jnp.float32)         # removed -> zero obs


def reset(cfg: JCfg, patch_mask, pos, facing, key):
    """Deterministic-init reset: caller supplies patch_mask, pos, facing (so it
    can mirror the NumPy oracle exactly). Berries start = patch_mask."""
    N, T, D = cfg.n_agents, cfg.n_berry_types, cfg.poison_delay
    s = State(
        berries=jnp.asarray(patch_mask, bool),
        pos=jnp.asarray(pos, jnp.int32),
        facing=jnp.asarray(facing, jnp.int32),
        marks=jnp.zeros((N, T), jnp.int32),
        pending=jnp.zeros((N, D + 1), bool),
        respawn=jnp.zeros(N, jnp.int32),
        patch_mask=jnp.asarray(patch_mask, bool),
        t=jnp.int32(0),
        key=key)
    return s, observe(cfg, s)


@partial(jax.jit, static_argnums=(0,))
def step(cfg: JCfg, s: State, actions):
    G, N, T = cfg.grid, cfg.n_agents, cfg.n_berry_types
    actions = jnp.asarray(actions, jnp.int32)
    rew = jnp.zeros(N, jnp.float32)

    # --- 1. delayed poison lands first
    poison_hits = s.pending[:, 0]
    rew = rew - cfg.r_poison * poison_hits.astype(jnp.float32)
    pending = jnp.concatenate([s.pending[:, 1:], jnp.zeros((N, 1), bool)], 1)

    active = s.respawn == 0

    # --- 2. movement, simultaneous; collisions cancel; removed excluded
    mv = (actions < 4) & active
    facing = jnp.where(mv, actions, s.facing)
    step_vec = _DELTA[jnp.clip(actions, 0, 3)] * mv[:, None]
    target = jnp.clip(s.pos + step_vec, 1, G - 2)
    keys = _cell_key(target, G)                             # (N,)
    # occupancy count over ACTIVE agents' target cells (inactive don't block)
    occ_count = jnp.zeros(G * G, jnp.int32).at[keys].add(active.astype(jnp.int32))
    unique = occ_count[keys] == 1
    ok = jnp.where(active, unique, True)                   # inactive: no-op move
    pos = jnp.where(ok[:, None], target, s.pos)

    # --- 3. eating (positions are distinct after collision -> no conflict)
    cell_berry = s.berries[:, pos[:, 0], pos[:, 1]].T       # (N, T) berry at each pos
    eat = (actions == 4) & active
    has_here = cell_berry.any(1) & eat
    eaten_t = jnp.argmax(cell_berry, axis=1)                # first True type
    did_eat = has_here
    rew = rew + cfg.r_eat * did_eat.astype(jnp.float32)
    onehot = jax.nn.one_hot(eaten_t, T, dtype=jnp.int32) * did_eat[:, None]
    marks = jnp.maximum(s.marks, onehot * cfg.mark_steps)
    # remove eaten berries (distinct positions -> scatter has no collision)
    berries = s.berries
    berries = berries.at[eaten_t, pos[:, 0], pos[:, 1]].set(
        jnp.where(did_eat, False, berries[eaten_t, pos[:, 0], pos[:, 1]]))
    # queue poison for type-0 eats
    ate0 = did_eat & (eaten_t == 0)
    pending = pending.at[:, -1].set(pending[:, -1] | ate0)

    # --- 4. zapping: scan agents in index order, mutating `alive` (order-exact)
    # vis_marked uses POST-eating marks (oracle computes it after step 3), so a
    # berry eaten this step can be enforced this same step.
    zap = (actions == 5) & active
    marked_mask = jnp.array(cfg.marked_mask)
    vis_marked = ((marks > 0) & marked_mask[None, :]).any(1)     # (N,)

    def zap_one(carry, i):
        alive, rew_c, respawn_c, n_land, n_marked = carry
        d = _DELTA[facing[i]]

        def scan_beam(bcarry, k):
            found, tgt = bcarry
            cell = pos[i] + d * (k + 1)
            hitmask = alive & (pos[:, 0] == cell[0]) & (pos[:, 1] == cell[1])
            any_hit = hitmask.any() & (~found)
            first = jnp.argmax(hitmask)                     # first alive at cell
            tgt = jnp.where(any_hit, first, tgt)
            found = found | (hitmask.any())
            return (found, tgt), None
        (found, tgt), _ = lax.scan(scan_beam, (False, 0), jnp.arange(cfg.zap_range))

        fires = zap[i]
        landed = fires & found
        rew_c = rew_c - jax.nn.one_hot(i, N) * (cfg.c_zap * fires)  # cost to zapper i
        rew_c = rew_c + jax.nn.one_hot(i, N) * (cfg.r_zap_bonus * landed)
        rew_c = rew_c - jax.nn.one_hot(tgt, N) * (cfg.c_zapped * landed)
        remove = landed & (cfg.zap_removal_steps > 0)
        respawn_c = jnp.where(jax.nn.one_hot(tgt, N, dtype=bool) & remove,
                              cfg.zap_removal_steps, respawn_c)
        alive = alive & ~(jax.nn.one_hot(tgt, N, dtype=bool) & remove)
        n_land = n_land + landed.astype(jnp.int32)
        n_marked = n_marked + (landed & vis_marked[tgt]).astype(jnp.int32)
        return (alive, rew_c, respawn_c, n_land, n_marked), None

    (alive, rew, respawn, zaps_landed, zaps_on_marked), _ = lax.scan(
        zap_one, (active, rew, s.respawn, jnp.int32(0), jnp.int32(0)),
        jnp.arange(N))

    # --- 5. regrowth
    key, kg = jax.random.split(s.key)
    grow = (s.patch_mask & ~berries) & (
        jax.random.uniform(kg, berries.shape) < cfg.regrow_prob)
    berries = berries | grow

    # --- 5b. respawn: tick timers; place respawned agents at free interior cells
    respawning = respawn == 1
    respawn = jnp.maximum(respawn - 1, 0)
    key, kr = jax.random.split(key)
    on_grid = (respawn == 0) & ~respawning
    occ_after = jnp.zeros((G, G), bool).at[pos[:, 0], pos[:, 1]].max(on_grid)
    interior = jnp.zeros((G, G), bool).at[1:G - 1, 1:G - 1].set(True)
    free = interior & ~berries.any(0) & ~occ_after          # (G,G) bool
    # pick, for each respawning agent, a distinct free cell via gumbel argmax
    flat_free = free.reshape(-1)
    def place_one(carry, i):
        taken, key_c, pos_c = carry
        key_c, ksub = jax.random.split(key_c)
        avail = flat_free & ~taken
        g = jax.random.gumbel(ksub, (G * G,)) + jnp.where(avail, 0., -1e9)
        cidx = jnp.argmax(g)
        newpos = jnp.array([cidx // G, cidx % G], jnp.int32)
        do = respawning[i]
        pos_c = pos_c.at[i].set(jnp.where(do, newpos, pos_c[i]))
        taken = taken.at[cidx].set(taken[cidx] | do)
        return (taken, key_c, pos_c), None
    (_, key, pos), _ = lax.scan(
        place_one, (jnp.zeros(G * G, bool), kr, pos), jnp.arange(N))
    key, kf = jax.random.split(key)
    facing = jnp.where(respawning, jax.random.randint(kf, (N,), 0, 4), facing)

    # --- 6. mark decay, clock
    marks = jnp.maximum(marks - 1, 0)
    t = s.t + 1
    ns = State(berries, pos, facing, marks, pending, respawn, s.patch_mask, t, key)
    done = t >= cfg.episode_len
    eats = jnp.array([jnp.sum(did_eat & (eaten_t == k)) for k in range(T)])
    info = dict(eats=eats,
                zaps_fired=jnp.sum(zap), zaps_landed=zaps_landed,
                zaps_on_marked=zaps_on_marked, poison_hits=jnp.sum(poison_hits),
                marked_agents=jnp.sum(vis_marked & active),
                active_agents=jnp.sum(active))
    return ns, observe(cfg, ns), rew, done, info



In [ ]:
%%writefile train_jax.py
"""
train_jax.py -- recurrent IPPO in JAX for berryworld_jax, fully jitted so it
runs on device and vmaps over seeds. Per-agent INDEPENDENT parameters (stacked
leading dim = pool size), so removing an agent is dropping a slice.

Validation discipline: at N=1, marked=(), this must reproduce Gate A (a lone
agent learns to avoid the poison berry). If it can't reproduce a result we
already have on CPU/PyTorch, the port is wrong -- don't spend GPU on it.

    python train_jax.py           # N=1 Gate A smoke on CPU
"""
from functools import partial
import numpy as np
import jax
import jax.numpy as jnp
from jax import lax
import flax.linen as nn
import optax

import berryworld_jax as bwj
from berryworld import BerryWorld, Config


# ------------------------------------------------------------------- network
class ACGRU(nn.Module):
    hidden: int
    n_actions: int

    @nn.compact
    def __call__(self, carry, obs):
        x = nn.tanh(nn.Dense(self.hidden)(obs))
        carry, h = nn.GRUCell(features=self.hidden)(carry, x)
        logits = nn.Dense(self.n_actions)(h)
        val = nn.Dense(1)(h)[..., 0]
        return carry, logits, val


# ------------------------------------------------------------- jittable reset
def _place(cfg, patch_mask, key):
    """Sample N distinct free interior cells (jittable, gumbel-masked)."""
    G, N = cfg.grid, cfg.n_agents
    interior = jnp.zeros((G, G), bool).at[1:G - 1, 1:G - 1].set(True)
    free = (interior & ~jnp.asarray(patch_mask).any(0)).reshape(-1)

    def pick(carry, _):
        taken, k = carry
        k, ks = jax.random.split(k)
        g = jax.random.gumbel(ks, (G * G,)) + jnp.where(free & ~taken, 0., -1e9)
        c = jnp.argmax(g)
        taken = taken.at[c].set(True)
        return (taken, k), c
    (_, _), cells = lax.scan(pick, (jnp.zeros(G * G, bool), key), None, length=N)
    pos = jnp.stack([cells // G, cells % G], axis=1).astype(jnp.int32)
    return pos


def reset_env(cfg, patch_mask, key):
    kp, kf, ks = jax.random.split(key, 3)
    pos = _place(cfg, patch_mask, kp)
    facing = jax.random.randint(kf, (cfg.n_agents,), 0, 4)
    s, obs = bwj.reset(cfg, jnp.asarray(patch_mask), pos, facing, ks)
    return s, obs


# --------------------------------------------------------------------- train
def make_train(cfg, patch_mask, hp):
    N, E = cfg.n_agents, hp["num_envs"]
    net = ACGRU(hp["hidden"], bwj.N_ACTIONS)
    obs_dim = 6 * (2 * cfg.view + 1) ** 2 + 2 + cfg.n_berry_types
    tx = optax.chain(optax.clip_by_global_norm(hp["max_grad"]),
                     optax.adam(hp["lr"]))

    vstep = jax.vmap(lambda s, a: bwj.step(cfg, s, a))       # over envs
    vreset = jax.vmap(lambda k: reset_env(cfg, patch_mask, k))

    def agent_apply(params, carry, obs):                    # obs (E,D) carry (E,H)
        return net.apply(params, carry, obs)
    # over agents: params axis 0, carry axis 0, obs axis 1(agent) -> outputs (N,E,*)
    fwd = jax.vmap(agent_apply, in_axes=(0, 0, 1))

    def train(rng):
        rng, ki = jax.random.split(rng)
        c0 = jnp.zeros((E, hp["hidden"]))
        o0 = jnp.zeros((E, obs_dim))
        params = jax.vmap(lambda k: net.init(k, c0, o0))(
            jax.random.split(ki, N))
        opt_state = tx.init(params)

        def update(runner, _):
            params, opt_state, rng = runner
            rng, kr = jax.random.split(rng)
            state, obs = vreset(jax.random.split(kr, E))     # obs (E,N,D)
            carry = jnp.zeros((N, E, hp["hidden"]))

            # --- rollout one episode across E envs via scan over time
            def step_t(carry_all, key_t):
                carry, state, obs = carry_all
                new_carry, logits, val = fwd(params, carry, obs)   # (N,E,*)
                key_t, ksa = jax.random.split(key_t)
                acts = jax.random.categorical(ksa, logits)         # (N,E)
                logp = jnp.take_along_axis(
                    jax.nn.log_softmax(logits), acts[..., None], -1)[..., 0]
                nstate, nobs, rew, done, info = vstep(state, acts.T)  # env wants (E,N)
                # store agent-major (N,E,*); obs came from env as (E,N,D)
                trans = (obs.transpose(1, 0, 2), acts, logp, val, rew.T,
                         info["eats"], info["zaps_landed"],
                         info["zaps_on_marked"], info["marked_agents"],
                         info["active_agents"])
                return (new_carry, nstate, nobs), trans

            rng, kt = jax.random.split(rng)
            (_, state, _), traj = lax.scan(
                step_t, (carry, state, obs),
                jax.random.split(kt, cfg.episode_len))
            (obs_t, act_t, logp_t, val_t, rew_t, eats_t,
             zl_t, zm_t, ma_t, aa_t) = traj                        # (T,N,E,*)

            # --- GAE per (agent, env)
            def gae_scan(carry, x):
                gae, next_v = carry
                rew, val = x
                delta = rew + hp["gamma"] * next_v - val
                gae = delta + hp["gamma"] * hp["lam"] * gae
                return (gae, val), gae
            _, adv = lax.scan(gae_scan, (jnp.zeros((N, E)), jnp.zeros((N, E))),
                              (rew_t, val_t), reverse=True)
            ret = adv + val_t
            adv = (adv - adv.mean()) / (adv.std() + 1e-8)

            # --- PPO update (independent per agent; single optimizer over stack)
            def loss_fn(params):
                def replay_agent(p, obs_a, act_a):            # (T,E,D),(T,E)
                    def rstep(carry, o):
                        carry, logits, val = net.apply(p, carry, o)
                        return carry, (logits, val)
                    _, (logits, val) = lax.scan(
                        rstep, jnp.zeros((E, hp["hidden"])), obs_a)
                    return logits, val
                logits, val = jax.vmap(replay_agent, in_axes=(0, 1, 1))(
                    params, obs_t, act_t)                     # (N,T,E,*)
                logits = logits.transpose(1, 0, 2, 3)         # (T,N,E,A)
                val = val.transpose(1, 0, 2)                  # (T,N,E)
                logp = jnp.take_along_axis(
                    jax.nn.log_softmax(logits), act_t[..., None], -1)[..., 0]
                ratio = jnp.exp(logp - logp_t)
                p1 = ratio * adv
                p2 = jnp.clip(ratio, 1 - hp["clip"], 1 + hp["clip"]) * adv
                pi_loss = -jnp.minimum(p1, p2).mean()
                v_loss = ((val - ret) ** 2).mean()
                probs = jax.nn.softmax(logits)
                ent = -(probs * jax.nn.log_softmax(logits)).sum(-1).mean()
                return pi_loss + hp["vf"] * v_loss - hp["ent"] * ent

            def ppo_epoch(carry, _):
                params, opt_state = carry
                g = jax.grad(loss_fn)(params)
                upd, opt_state = tx.update(g, opt_state, params)
                params = optax.apply_updates(params, upd)
                return (params, opt_state), None
            (params, opt_state), _ = lax.scan(
                ppo_epoch, (params, opt_state), None, length=hp["epochs"])

            # eats_t is (T, E, n_berry_types); per-env episode totals, env-mean
            zl, zm = zl_t.sum(), zm_t.sum()
            prev = ma_t.sum() / jnp.maximum(aa_t.sum(), 1)      # marked prevalence
            share = zm / jnp.maximum(zl, 1)                     # zaps hitting marked
            metrics = dict(eat0=eats_t[..., 0].sum() / E,
                           eat1=eats_t[..., 1].sum() / E,
                           ret=rew_t.sum(0).mean(),
                           selectivity=share / jnp.maximum(prev, 1e-8),
                           zaps=zl / E)                         # landed zaps/episode
            return (params, opt_state, rng), metrics

        (params, _, _), metrics = lax.scan(
            update, (params, opt_state, rng), None, length=hp["updates"])
        return params, metrics

    return train


DEFAULT_HP = dict(hidden=64, lr=3e-4, gamma=0.99, lam=0.95, clip=0.2,
                  epochs=3, ent=0.01, vf=0.5, max_grad=0.5,
                  num_envs=16, updates=400)


def build_patch_mask(marked, n_agents, seed=0):
    c = Config(); c.marked_berries = marked; c.n_agents = n_agents
    return BerryWorld(c, seed=seed).patch_mask.copy()


if __name__ == "__main__":
    import time
    marked = ()
    cfg = bwj.JCfg(n_agents=1, episode_len=300, poison_delay=25,
                   zap_removal_steps=25,
                   marked_mask=tuple(t in marked for t in range(2)))
    pm = build_patch_mask(marked, 1)
    train = make_train(cfg, pm, DEFAULT_HP)
    t0 = time.time()
    params, metrics = jax.block_until_ready(jax.jit(train)(jax.random.PRNGKey(0)))
    dt = time.time() - t0
    e0 = np.array(metrics["eat0"])
    print(f"Gate A (N=1, no marks) -- {dt:.1f}s for {DEFAULT_HP['updates']} updates")
    print(f"  poison eaten/episode: first10 {e0[:10].mean():.1f} -> last10 {e0[-10:].mean():.1f}")
    print("  PASS (avoidance learned)" if e0[-10:].mean() < 0.6 * e0[:10].mean()
          else "  (no clear avoidance -- inspect)")



In [ ]:
%%writefile run_sweep.py
"""
run_sweep.py -- the faithful-scale population sweep, driver for GPU (Colab).

For each (N, condition) cell it vmaps the JAX IPPO train over seeds and logs
per-update metrics. The question this answers: does enforcement (selectivity>1)
fire when the population grows, given enough training?

Local (CPU) smoke:  python run_sweep.py --smoke
Faithful (GPU):     imported from the Colab notebook with a large hp.

NOTE / honest limitations of this diagnostic version:
  * All N agents play every episode (Koster's 8-of-12 per-episode sampling is a
    refinement not yet implemented -- this tests population SIZE, the primary
    lever).
  * patch_mask is fixed per (N,condition) cell across seeds; seed varies net
    init + rollout + resets. Per-seed patches are the rigorous version, later.
"""
import argparse
import csv
import jax
import numpy as np
import train_jax as T
import berryworld_jax as bwj


def env_variant(poison_delay=25, r_zap_bonus=0.0,
                episode_len=300, zap_removal_steps=25):
    return dict(poison_delay=poison_delay, r_zap_bonus=r_zap_bonus,
                episode_len=episode_len, zap_removal_steps=zap_removal_steps)


def run_sweep(n_list, conditions, n_seeds, hp, env_list, out_csv="sweep.csv"):
    """Sweep over N x env_variant x condition. Each env_variant carries its own
    poison_delay (difficulty) and r_zap_bonus (enforcement incentive), both
    logged per row so the D and bonus axes are analysable."""
    if isinstance(env_list, dict):                 # allow a single env
        env_list = [env_list]
    rows = []
    for N in n_list:
        for env in env_list:
            for marked in conditions:
                cfg = bwj.JCfg(
                    n_agents=N, episode_len=env["episode_len"],
                    poison_delay=env["poison_delay"],
                    zap_removal_steps=env["zap_removal_steps"],
                    r_zap_bonus=env["r_zap_bonus"],
                    marked_mask=tuple(t in marked for t in range(2)))
                pm = T.build_patch_mask(marked, N)
                train1 = T.make_train(cfg, pm, hp)
                metrics_fn = jax.jit(jax.vmap(lambda k: train1(k)[1]))
                keys = jax.random.split(jax.random.PRNGKey(0), n_seeds)
                m = jax.block_until_ready(metrics_fn(keys))        # each (S, U)
                cond = "".join(str(b) for b in marked) or "none"
                U = m["eat0"].shape[1]
                e = np.asarray(m["eat0"]); sel = np.asarray(m["selectivity"])
                print(f"N={N:2d} D={env['poison_delay']:2d} "
                      f"bonus={env['r_zap_bonus']:.1f} {cond:4s}  "
                      f"eat0 {e[:, :5].mean():4.0f}->{e[:, -5:].mean():4.0f}"
                      f"  sel {sel[:, -5:].mean():.2f}  (n={n_seeds})", flush=True)
                for s in range(n_seeds):
                    for u in range(U):
                        rows.append(dict(
                            N=N, D=env["poison_delay"],
                            bonus=env["r_zap_bonus"], condition=cond,
                            seed=s, update=u,
                            eat0=float(m["eat0"][s, u]), eat1=float(m["eat1"][s, u]),
                            selectivity=float(m["selectivity"][s, u]),
                            zaps=float(m["zaps"][s, u]), ret=float(m["ret"][s, u])))
    with open(out_csv, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        w.writeheader(); w.writerows(rows)
    print(f"wrote {len(rows)} rows -> {out_csv}", flush=True)
    return rows


ENV = env_variant()                                # D=25, bonus=0 default

# Faithful-scale hyperparameters for GPU. steps = updates * num_envs * episode_len.
# updates=1500, num_envs=256 -> ~1.15e8 steps/run (Koster regime is 2-4e8).
FAITHFUL_HP = dict(hidden=64, lr=3e-4, gamma=0.99, lam=0.95, clip=0.2,
                   epochs=3, ent=0.01, vf=0.5, max_grad=0.5,
                   num_envs=256, updates=1500)


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--smoke", action="store_true", help="tiny CPU run")
    a = ap.parse_args()
    if a.smoke:
        hp = dict(FAITHFUL_HP); hp["num_envs"] = 8; hp["updates"] = 20
        envs = [env_variant(poison_delay=d) for d in (25, 75)]     # D axis
        run_sweep([12], [(), (0,), (0, 1)], n_seeds=2, hp=hp, env_list=envs,
                  out_csv="sweep_smoke.csv")
    else:
        run_sweep([10, 12], [(), (0,), (0, 1)], n_seeds=8,
                  hp=FAITHFUL_HP, env_list=[ENV], out_csv="sweep.csv")



## Sanity: GPU visible + one tiny run on device

In [ ]:
import jax; print('devices:', jax.devices())
import run_sweep as R
hp = dict(R.FAITHFUL_HP); hp['num_envs'] = 16; hp['updates'] = 5
_ = R.run_sweep([12], [(0,)], n_seeds=2, hp=hp,
                env_list=[R.env_variant(poison_delay=75)], out_csv='sanity.csv')
print('sanity OK')

## Quick D-sweep (fast, ~2-3 min) -- confirm the flow, get a first read
D in {25,50,75}, N=12, 2 seeds, 300 updates. Bump to the full run below once this works.

In [7]:
import run_sweep as R
hp = dict(R.FAITHFUL_HP); hp['num_envs'] = 32; hp['updates'] = 300
envs = [R.env_variant(poison_delay=d) for d in (25, 50, 75)]
rows = R.run_sweep([12], [(), (0,), (0, 1)], n_seeds=2,
                   hp=hp, env_list=envs, out_csv='dsweep.csv')
print('DONE - dsweep.csv written')

N=12 D=25 bonus=0.0 none  eat0   61->   5  sel 0.00  (n=2)
N=12 D=25 bonus=0.0 0     eat0   60->   4  sel 0.90  (n=2)
N=12 D=25 bonus=0.0 01    eat0   60->   4  sel 0.99  (n=2)
N=12 D=50 bonus=0.0 none  eat0   61->  35  sel 0.00  (n=2)
N=12 D=50 bonus=0.0 0     eat0   61->  27  sel 0.81  (n=2)
N=12 D=50 bonus=0.0 01    eat0   61->  28  sel 0.97  (n=2)
N=12 D=75 bonus=0.0 none  eat0   61->  83  sel 0.00  (n=2)
N=12 D=75 bonus=0.0 0     eat0   61->  80  sel 0.81  (n=2)


KeyboardInterrupt: 

## Plot + verdict

In [8]:
import csv, numpy as np, matplotlib.pyplot as plt
rows = list(csv.DictReader(open('dsweep.csv')))
Ds = sorted({int(r['D']) for r in rows}); conds = ['none','0','01']
lab = {'none':'() no rule','0':'(0,) important','01':'(0,1) silly'}
def lastq(sub):
    mx = max(int(r['update']) for r in sub)
    return [r for r in sub if int(r['update']) >= 0.75*mx]
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
for c in conds:
    ys = [np.mean([float(r['eat0']) for r in lastq([r for r in rows if r['condition']==c and int(r['D'])==D])]) for D in Ds]
    ax[0].plot(Ds, ys, 'o-', label=lab[c])
ax[0].set_xlabel('poison_delay D'); ax[0].set_ylabel('eat0 (last-quarter)')
ax[0].set_title('avoidance vs difficulty -- gap = Koster effect'); ax[0].legend()
Dh = Ds[-1]
for c in conds:
    sub = [r for r in rows if r['condition']==c and int(r['D'])==Dh]
    U = max(int(r['update']) for r in sub)+1
    y = np.array([[float(r['selectivity']) for r in sub if int(r['update'])==u] for u in range(U)])
    ax[1].plot(y.mean(1), label=lab[c])
ax[1].axhline(1.0, color='k', ls='--', lw=1)
ax[1].set_title(f'enforcement selectivity at D={Dh} (>1 = fired)'); ax[1].set_xlabel('update'); ax[1].legend()
plt.tight_layout(); plt.savefig('dsweep.png', dpi=110); plt.show()
print('\nlast-quarter eat0 (gap of () above (0,) = effect):')
for D in Ds:
    v = {c: np.mean([float(r['eat0']) for r in lastq([r for r in rows if r['condition']==c and int(r['D'])==D])]) for c in conds}
    s = {c: np.mean([float(r['selectivity']) for r in lastq([r for r in rows if r['condition']==c and int(r['D'])==D])]) for c in conds}
    gap = v['none']-v['0']; tag = 'EFFECT' if gap > 2 else 'flat'
    print(f"  D={D:2d}  ()={v['none']:5.1f}  (0,)={v['0']:5.1f}  (0,1)={v['01']:5.1f}  gap={gap:+5.1f} [{tag}]  sel(0,)={s['0']:.2f}")

FileNotFoundError: [Errno 2] No such file or directory: 'dsweep.csv'

## Full run (slower) -- only after the quick one looks right
4 seeds, 800 updates. Re-run the plot cell after this to get the publication-scale read.

In [ ]:
import run_sweep as R
hp = dict(R.FAITHFUL_HP); hp['num_envs'] = 64; hp['updates'] = 800
envs = [R.env_variant(poison_delay=d) for d in (25, 50, 75)]
rows = R.run_sweep([12], [(), (0,), (0, 1)], n_seeds=4,
                   hp=hp, env_list=envs, out_csv='dsweep.csv')
print('DONE - full dsweep.csv written')

## Bonus axis: does a zap reward revive enforcement at D=75?
Run only if difficulty opened a niche but `(0,)` still didn't separate.

In [ ]:
import run_sweep as R
hp = dict(R.FAITHFUL_HP); hp['num_envs'] = 64; hp['updates'] = 800
envs = [R.env_variant(poison_delay=75, r_zap_bonus=b) for b in (0.0, 0.5)]
rows2 = R.run_sweep([12], [(), (0,), (0, 1)], n_seeds=4,
                    hp=hp, env_list=envs, out_csv='dsweep_bonus.csv')
import csv, numpy as np
rr = list(csv.DictReader(open('dsweep_bonus.csv')))
print('\nD=75, does bonus revive enforcement? (selectivity>1 = fired)')
for b in ('0.0','0.5'):
    for c in ('none','0','01'):
        sub=[r for r in rr if r['condition']==c and r['bonus']==b]
        if not sub: continue
        mx=max(int(r['update']) for r in sub); lq=[r for r in sub if int(r['update'])>=0.75*mx]
        print(f"  bonus={b} {c:4s}  sel {np.mean([float(r['selectivity']) for r in lq]):.2f}  eat0 {np.mean([float(r['eat0']) for r in lq]):.1f}")

## Download results

In [ ]:
import os
from google.colab import files
for f in ("dsweep.csv", "dsweep.png", "dsweep_bonus.csv"):
    if os.path.exists(f):
        print("downloading", f)
        files.download(f)
    else:
        print("skip (not found):", f)
